In [150]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
from sklearn.pipeline import Pipeline
import plotly.express as px
import missingno as msno
from data_profiling import ProfileReport

In [151]:
df = sns.load_dataset("titanic")

### Challenge 1 : NumPy (l'essentiel pour l'EDA)
#### Travailler sur une matrice 5×4 contenant les entiers de 0 à 19

In [152]:
arr = np.arange(20).reshape(5,4)
arr

array([[ 0,  1,  2,  3],
       [ 4,  5,  6,  7],
       [ 8,  9, 10, 11],
       [12, 13, 14, 15],
       [16, 17, 18, 19]])

### 1. Donner sa forme, son nombre de dimensions, son type et sa taille totale

In [153]:
arr.shape, arr.ndim, arr.dtype, arr.size

((5, 4), 2, dtype('int64'), 20)

### 2. Accéder à un seul élément (ligne d'indice 3, colonne d'indice 2), puis au dernier élément avec des indices négatifs.

In [154]:
arr[3,2], arr[-1,-1], 

(np.int64(14), np.int64(19))

### 3. Extraire la première ligne, la dernière colonne, puis la deuxième colonne entière.

In [155]:
arr[0], arr[:,-1] , arr[:,1]

(array([0, 1, 2, 3]), array([ 3,  7, 11, 15, 19]), array([ 1,  5,  9, 13, 17]))

### 4. Extraire par slicing le bloc central (lignes d'indice 1 à 3, colonnes d'indice 1 à 2)

In [156]:
arr[1:3,1:2]

array([[5],
       [9]])

### 5. Extraire une ligne sur deux, puis inverser l'ordre des lignes

In [157]:
arr[::2][::-1]

array([[16, 17, 18, 19],
       [ 8,  9, 10, 11],
       [ 0,  1,  2,  3]])

### 6. Sélectionner les lignes 0, 2 et 4 avec une liste d'indices. Quelle différence avec un slicing ?


In [158]:
arr[[0,2,4]]

array([[ 0,  1,  2,  3],
       [ 8,  9, 10, 11],
       [16, 17, 18, 19]])

### 7. Avec un masque booléen, sélectionner les valeurs paires supérieures à 8, puis les remplacer par 0.


In [159]:
arr[arr % 2 == 0] = 0
arr

array([[ 0,  1,  0,  3],
       [ 0,  5,  0,  7],
       [ 0,  9,  0, 11],
       [ 0, 13,  0, 15],
       [ 0, 17,  0, 19]])

### 8. Retrouver les positions (indices) des valeurs divisibles par 5.


In [160]:
indices = np.argwhere(arr % 5 == 0)
for idx in indices:
    print(f"element: [{arr[idx[0],idx[1]]}] => {idx}")

element: [0] => [0 0]
element: [0] => [0 2]
element: [0] => [1 0]
element: [5] => [1 1]
element: [0] => [1 2]
element: [0] => [2 0]
element: [0] => [2 2]
element: [0] => [3 0]
element: [0] => [3 2]
element: [15] => [3 3]
element: [0] => [4 0]
element: [0] => [4 2]


### 9. Vue ou copie : modifier une valeur dans un slice et vérifier si la matrice d'origine change.

In [161]:
arrcopy = arr.copy()
arrcopy[1:3,2:4] = 0
arrcopy != arr

array([[False, False, False, False],
       [False, False, False,  True],
       [False, False, False,  True],
       [False, False, False, False],
       [False, False, False, False]])

### Partie B : NumPy sur le Titanic


In [162]:
age = df["age"].to_numpy()
fare = df["fare"].to_numpy()

### 2. Compter les valeurs manquantes de age. Comparer la moyenne simple et la moyenne qui ignore les NaN : que se passe-t-

In [163]:
manquantes = np.isnan(age).sum()
moyenne_simple = np.mean(age)
moyenne_ignore_NAN_age = np.nanmean(age)
moyenne_ignore_NAN_age , moyenne_simple

(np.float64(29.69911764705882), np.float64(nan))

### 3. Calculer moyenne, médiane, écart-type, minimum, maximum et percentiles 25 et 75 de age et de fare. Comparer moyenne et médiane de fare : que révèle l'écart ?

In [164]:
moyenne_ignore_NAN_fare = np.nanmean(fare)
median_ignore_nan_fare = np.nanmedian(fare)
median_ignore_nan_age = np.nanmedian(age)
std_ignore_NAN_age = np.nanstd(age)
std_ignore_NAN_fare = np.nanstd(fare)
min_ignore_NAN_age = np.nanmin(age)
min_ignore_NAN_fare = np.nanmin(fare)
max_ignore_NAN_age= np.nanmax(age)
max_ignore_NAN_fare = np.nanmax(fare)

Q1_fare = np.nanpercentile(fare, 25)
Q3_fare = np.nanpercentile(fare, 75)
Q1_age = np.nanpercentile(age, 25)
Q3_age = np.nanpercentile(age, 75)

### 4. Calculer l'IQR de fare, la borne supérieure Q3 + 1.5 × IQR, puis compter les outliers avec un masque.

In [165]:
iqr_fare = Q3_fare - Q1_fare
upper_fare = Q3_fare + iqr_fare * 1.5
lower_fare = Q1_fare - iqr_fare * 1.5
outliers_fare = (fare > upper_fare) & (fare < lower_fare)

iqr_age = Q3_age - Q1_age
upper_age = Q3_age + iqr_age * 1.5
lower_age = Q1_age - iqr_age * 1.5
outliers_age = (age > upper_age) & (age < lower_age)

age[outliers_age].sum() ,fare[outliers_fare].sum()

(np.float64(0.0), np.float64(0.0))

### 5. Appliquer np.log1p à fare et comparer la distribution avant et après (moyenne, médiane).

In [166]:
new_distribution = np.log1p(fare)
np.nanstd(new_distribution),np.nanstd(fare)

(np.float64(0.968504319324457), np.float64(49.6655344447741))

### 6. Créer un tableau « enfant / adulte » (seuil de 12 ans) et compter les enfants

In [167]:
label = np.where(age > 12, "adult", "enfant")
count_label = (label == "enfant").sum()

### 7. Calculer la corrélation entre age et fare (attention aux NaN) et l'interpréter.

In [168]:
mask_nan = np.isnan(age) | np.isnan(fare)
age = age[~mask_nan]
fare = fare[~mask_nan]
np.corrcoef(age,fare)

array([[1.        , 0.09606669],
       [0.09606669, 1.        ]])

## Challenge 2 : Pandas (l'essentiel pour l'EDA)

### 1. Extraire la colonne age du Titanic sous forme de Series. 
- Quelle différence avec l'extraction de age sous forme de DataFrame (double crochets) ?

In [169]:
print(type(df["age"]), type(df[["age"]]))
print(df["age"].shape, df[["age"]].shape)
print(df["age"].ndim, df[["age"]].ndim)

<class 'pandas.core.series.Series'> <class 'pandas.core.frame.DataFrame'>
(891,) (891, 1)
1 2


### 2. Accéder à des éléments avec loc (étiquette) et iloc (position). Donner un cas où les deux donnent des résultats différen

In [170]:
df["age"].loc[1], df["age"].iloc[20]


(np.float64(38.0), np.float64(35.0))

 ### 3. Faire des opérations vectorisées : fare en centaines, age arrondi, somme de sibsp et parch.

In [171]:
# df["fare"] / 1 ,df["age"].round(2), df["sibsp"] + df["parch"]

## 4. Explorer une Series : unique, nunique, value_counts (avec et sans normalize), min, max, idxmax,
- sort_values, nlargest.

In [172]:
age = df["age"]
print(f"unique: {age.unique()}, Nunique: {age.nunique()},Value_coutns: {age.value_counts()}, Min: {age.min()}, Max: {age.max()}, Idxmax: {age.idxmax()}, Sort_values: {age.sort_values()}, Nlargest: {age.nlargest()}")

unique: [22.   38.   26.   35.     nan 54.    2.   27.   14.    4.   58.   20.
 39.   55.   31.   34.   15.   28.    8.   19.   40.   66.   42.   21.
 18.    3.    7.   49.   29.   65.   28.5   5.   11.   45.   17.   32.
 16.   25.    0.83 30.   33.   23.   24.   46.   59.   71.   37.   47.
 14.5  70.5  32.5  12.    9.   36.5  51.   55.5  40.5  44.    1.   61.
 56.   50.   36.   45.5  20.5  62.   41.   52.   63.   23.5   0.92 43.
 60.   10.   64.   13.   48.    0.75 53.   57.   80.   70.   24.5   6.
  0.67 30.5   0.42 34.5  74.  ], Nunique: 88,Value_coutns: age
24.00    30
22.00    27
18.00    26
28.00    25
30.00    25
         ..
24.50     1
0.67      1
0.42      1
34.50     1
74.00     1
Name: count, Length: 88, dtype: int64, Min: 0.42, Max: 80.0, Idxmax: 630, Sort_values: 803    0.42
755    0.67
644    0.75
469    0.75
78     0.83
       ... 
859     NaN
863     NaN
868     NaN
878     NaN
888     NaN
Name: age, Length: 891, dtype: float64, Nlargest: 630    80.0
851    74.0
96     

### 5. Détecter et traiter les manquants d'une Series : isna, notna, fillna, dropna.

In [173]:
mask_na = age.isna()
mask_notna = age.notna()
age.fillna(age.notna().median())
Class = df["class"].dropna()

### 6. Transformer : astype, map avec un dictionnaire (par exemple male en 0 et female en 1), apply avec une petite fonction.

In [174]:
age.fillna(age.notna().median()).astype(np.int32)
sex = {
    "male":0,
    "female":1
}
df["sex"].map(sex)
Class.apply(lambda name: name.capitalize())

0       Third
1       First
2       Third
3       First
4       Third
        ...  
886    Second
887     First
888     Third
889     First
890     Third
Name: class, Length: 891, dtype: category
Categories (3, object): ['First', 'Second', 'Third']

## Partie B : conditions logiques et filtres
### 1. Écrire une condition simple (age supérieur à 30) et observer le résultat. Combien de True ? Quelle
- proportion ?

In [175]:
age = age.notna()
(age > 30).sum(), (age > 30).mean()

(np.int64(0), np.float64(0.0))

### 3. Utiliser isin, between, isna, notna pour écrire : passagers embarqués à C ou Q, tarif entre 10 et 50, âge manquant

In [176]:
df["embarked"].isin(["c"]), df["fare"].between(1,50), df["age"].isna()

(0      False
 1      False
 2      False
 3      False
 4      False
        ...  
 886    False
 887    False
 888    False
 889    False
 890    False
 Name: embarked, Length: 891, dtype: bool,
 0       True
 1      False
 2       True
 3      False
 4       True
        ...  
 886     True
 887     True
 888     True
 889     True
 890     True
 Name: fare, Length: 891, dtype: bool,
 0      False
 1      False
 2      False
 3      False
 4      False
        ...  
 886    False
 887    False
 888     True
 889    False
 890    False
 Name: age, Length: 891, dtype: bool)

### 3. Filtrer le DataFrame avec le masque, puis avec loc (masque + choix de colonnes), puis avec query.
- Comparer les trois écritures.

In [192]:
age_mask = df["age"] > 10
df.loc[age_mask, ["age", "class"]]
df.query("@age_mask")

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
885,0,3,female,39.0,0,5,29.1250,Q,Third,woman,False,NaN,Queenstown,no,False
886,0,2,male,27.0,0,0,13.0000,S,Second,man,True,NaN,Southampton,no,True
887,1,1,female,19.0,0,0,30.0000,S,First,woman,False,B,Southampton,yes,True
889,1,1,male,26.0,0,0,30.0000,C,First,man,True,C,Cherbourg,yes,True


### 5. Compter et mesurer avec un masque : somme = nombre de lignes, moyenne = proportion. Calculer
- ainsi le taux de survie global.

In [193]:
survived_mask = df["survived"] == 1

survivor_count = survived_mask.sum()
survival_rate = survived_mask.mean()

print("Number of survivors:", survivor_count)
print("Survival rate:", survival_rate)
print("Survival percentage:", survival_rate * 100)

Number of survivors: 342
Survival rate: 0.3838383838383838
Survival percentage: 38.38383838383838


### 6. Créer une colonne conditionnelle avec np.where (par exemple enfant si age < 12), puis une version à
- plusieurs cas avec np.select

In [194]:
df["age_category"] = np.where(
    df["age"] < 12,
    "child",
    "adult"
)

df[["age", "age_category"]].head(10)

,age,age_category
0,22.0,adult
1,38.0,adult
2,26.0,adult
3,35.0,adult
4,35.0,adult
5,NaN,adult
6,54.0,adult
7,2.0,child
8,27.0,adult
9,14.0,adult


### 7. Répondre avec des conditions logiques : combien de femmes en 1re classe ? Quel est leur taux de
- survie ? Combien d'hommes de plus de 60 ans ? Combien de passagers voyageant seuls en 3e classe ?

In [195]:
women_first_class = (
    (df["sex"] == "female") &
    (df["pclass"] == 1)
)

women_first_class_count = women_first_class.sum()

print("Women in first class:", women_first_class_count)

Women in first class: 94


In [178]:
df.head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


In [179]:
df.dtypes

survived          int64
pclass            int64
sex              object
age             float64
sibsp             int64
parch             int64
fare            float64
embarked         object
class          category
who              object
adult_male         bool
deck           category
embark_town      object
alive            object
alone              bool
dtype: object

In [180]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 15 columns):
 #   Column       Non-Null Count  Dtype   
---  ------       --------------  -----   
 0   survived     891 non-null    int64   
 1   pclass       891 non-null    int64   
 2   sex          891 non-null    object  
 3   age          714 non-null    float64 
 4   sibsp        891 non-null    int64   
 5   parch        891 non-null    int64   
 6   fare         891 non-null    float64 
 7   embarked     889 non-null    object  
 8   class        891 non-null    category
 9   who          891 non-null    object  
 10  adult_male   891 non-null    bool    
 11  deck         203 non-null    category
 12  embark_town  889 non-null    object  
 13  alive        891 non-null    object  
 14  alone        891 non-null    bool    
dtypes: bool(2), category(2), float64(2), int64(4), object(5)
memory usage: 80.7+ KB


In [181]:
df.describe(include=["object","category"])

,sex,embarked,class,who,deck,embark_town,alive
count,891,889,891,891,203,889,891
unique,2,3,3,3,7,3,2
top,male,S,Third,man,C,Southampton,no
freq,577,644,491,537,59,644,549


In [182]:
df.describe()

,survived,pclass,age,sibsp,parch,fare
count,891.000000,891.000000,714.000000,891.000000,891.000000,891.000000
mean,0.383838,2.308642,29.699118,0.523008,0.381594,32.204208
std,0.486592,0.836071,14.526497,1.102743,0.806057,49.693429
min,0.000000,1.000000,0.420000,0.000000,0.000000,0.000000
25%,0.000000,2.000000,20.125000,0.000000,0.000000,7.910400
50%,0.000000,3.000000,28.000000,0.000000,0.000000,14.454200
75%,1.000000,3.000000,38.000000,1.000000,0.000000,31.000000
max,1.000000,3.000000,80.000000,8.000000,6.000000,512.329200


In [183]:
columns = df.columns.to_list()
result = pd.DataFrame({col:df[col].value_counts() for col in columns})